In [1]:
!pip install -q kaggle transformers accelerate librosa


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
!{sys.executable} -m pip install --user "numpy<2"


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import os

# Kaggle-Zugangsdaten hier eintragen
os.environ["KAGGLE_USERNAME"] = "DEIN_USERNAME"
os.environ["KAGGLE_KEY"]      = "DEIN_API_KEY"

# RAVDESS Speech-Audio auf Kaggle
DATASET_SLUG = "uwrfkaggler/ravdess-emotional-speech-audio"
DOWNLOAD_DIR = "/tmp/ravdess"

# Prompt-Modus: 'normal' oder 'prosodic'
PROMPT_MODE = "normal"

# Dataset von Kaggle herunterladen und entpacken
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --unzip

Dataset URL: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
License(s): CC-BY-NC-SA-4.0
100%|████████████████████████████████████████| 429M/429M [00:17<00:00, 25.1MB/s]



In [2]:
from pathlib import Path
from collections import Counter

# RAVDESS Filename-Encoding (7 Felder, bindestrichgetrennt):
#   03-01-06-01-02-01-12.wav
#   Feld 3 = Emotion: 01 neutral, 02 calm, 03 happy, 04 sad,
#                     05 angry, 06 fearful, 07 disgust, 08 surprised
RAVDESS_EMOTION = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

def ravdess_emotion(path):
    parts = path.stem.split("-")
    return RAVDESS_EMOTION.get(parts[2]) if len(parts) >= 3 else None

all_files = sorted(Path(DOWNLOAD_DIR).rglob("*.wav"))
print(f"Gefundene WAV-Dateien: {len(all_files)}")
for f in all_files[:5]:
    print(f"  {f}  [{ravdess_emotion(f)}]")

gt_counts = Counter(ravdess_emotion(f) for f in all_files)
print(f"\nGround-truth Verteilung: {gt_counts.most_common()}")

Gefundene WAV-Dateien: 2880
  /tmp/ravdess/Actor_01/03-01-01-01-01-01-01.wav  [neutral]
  /tmp/ravdess/Actor_01/03-01-01-01-01-02-01.wav  [neutral]
  /tmp/ravdess/Actor_01/03-01-01-01-02-01-01.wav  [neutral]
  /tmp/ravdess/Actor_01/03-01-01-01-02-02-01.wav  [neutral]
  /tmp/ravdess/Actor_01/03-01-02-01-01-01-01.wav  [calm]

Ground-truth Verteilung: [('calm', 384), ('happy', 384), ('sad', 384), ('angry', 384), ('fearful', 384), ('disgust', 384), ('surprised', 384), ('neutral', 192)]


In [3]:
import json
import torch
import librosa
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

# Prompt-Set: identisch zur EMO-DB-Version, um den Prompt-Effekt
# sprachenuebergreifend vergleichbar zu halten.
# Hinweis: RAVDESS enthaelt calm und surprised, die im Label-Set nicht
# vorkommen — diese Samples werden bei der Auswertung separat behandelt.
PROMPTS = {
    "normal": (
        "Listen to this audio clip and classify the emotion of the speaker. "
        "Choose from: neutral, calm, happy, sad, angry, fearful, disgust, surprised "
        "Reply with just the emotion label."
    ),
    "prosodic": (
        "Listen to this audio clip and classify the emotion of the speaker "
        "based only on prosodic and acoustic features such as pitch, "
        "speech rate, rhythm and loudness. "
        "Ignore the spoken words and linguistic content entirely. "
        "Choose from: anger, anxiety, boredom, disgust, fear, happiness, neutral, sadness. "
        "Reply with just the emotion label."
    ),
}

PROMPT = PROMPTS[PROMPT_MODE]
output_file = f"emotion_results_qwen_ravdess_{PROMPT_MODE}.json"

# Resume
results = {}
if os.path.exists(output_file):
    with open(output_file) as f:
        results = json.load(f)
    print(f"{len(results)} Dateien bereits verarbeitet")

# Modell laden
print("Lade Modell...")
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
SR = processor.feature_extractor.sampling_rate
print(f"Modell geladen auf: {next(model.parameters()).device}")

2026-05-05 15:35:48.473227: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-05 15:35:48.500421: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-05 15:35:48.500444: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-05 15:35:48.501120: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 15:35:48.505451: I tensorflow/core/platform/cpu_feature_guar

Lade Modell...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Modell geladen auf: cuda:0


In [30]:
all_files[0].name

'03-01-01-01-01-01-01.wav'

In [29]:
for i in range(len(all_files)):
    if '03-01-01-01-01-01-01.wav'==all_files[i].name:
        print(1)
    

1
1


In [ ]:
total_files = len(all_files)

for i, audio_file in enumerate(all_files, 1):
    if audio_file.name in results:
        print(f"({i}/{total_files}) Uebersprungen: {audio_file.name}")
        continue

    print(f"({i}/{total_files}) {audio_file.name}", end=" ... ")
    try:
        audio, _ = librosa.load(str(audio_file), sr=SR)

        conversation = [{"role": "user", "content": [
            {"type": "audio", "audio_url": str(audio_file)},
            {"type": "text",  "text": PROMPT},
        ]}]

        text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, audios=[audio], sampling_rate=SR, return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        generate_ids = model.generate(**inputs, max_new_tokens=10)
        generate_ids = generate_ids[:, inputs["input_ids"].size(1):]
        emotion = processor.batch_decode(generate_ids, skip_special_tokens=True)[0].strip()

        results[audio_file.name] = emotion
        print(emotion)

        with open(output_file, "w") as f:
            json.dump(results, f, indent=2)

    except Exception as e:
        print(f"FEHLER: {e}")

print(f"\nFertig. {len(results)}/{total_files} Dateie

In [4]:
total_files = len(all_files)

for i, audio_file in enumerate(all_files, 1):
    if audio_file.name in results:
        print(f"({i}/{total_files}) Uebersprungen: {audio_file.name}")
        continue

    print(f"({i}/{total_files}) {audio_file.name}", end=" ... ")
    try:
        audio, _ = librosa.load(str(audio_file), sr=SR)

        conversation = [{"role": "user", "content": [
            {"type": "audio", "audio_url": str(audio_file)},
            {"type": "text",  "text": PROMPT},
        ]}]

        text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, audios=[audio], sampling_rate=SR, return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        generate_ids = model.generate(**inputs, max_new_tokens=10)
        generate_ids = generate_ids[:, inputs["input_ids"].size(1):]
        emotion = processor.batch_decode(generate_ids, skip_special_tokens=True)[0].strip()

        results[audio_file.name] = emotion
        print(emotion)

        with open(output_file, "w") as f:
            json.dump(results, f, indent=2)

    except Exception as e:
        print(f"FEHLER: {e}")

print(f"\nFertig. {len(results)}/{total_files} Dateien verarbeitet.")

(1/2880) 03-01-01-01-01-01-01.wav ... 

Keyword argument `audios` is not a valid argument for this processor and will be ignored.


Angry
(2/2880) 03-01-01-01-01-02-01.wav ... Neutral
(3/2880) 03-01-01-01-02-01-01.wav ... Angry
(4/2880) 03-01-01-01-02-02-01.wav ... Neutral
(5/2880) 03-01-02-01-01-01-01.wav ... Angry
Angry80) 03-01-02-01-01-02-01.wav ... 
(7/2880) 03-01-02-01-02-01-01.wav ... Happy
(8/2880) 03-01-02-01-02-02-01.wav ... Happy
(9/2880) 03-01-02-02-01-01-01.wav ... Angry
(10/2880) 03-01-02-02-01-02-01.wav ... Neutral
Angry880) 03-01-02-02-02-01-01.wav ... 
(12/2880) 03-01-02-02-02-02-01.wav ... Happy
(13/2880) 03-01-03-01-01-01-01.wav ... Neutral
(14/2880) 03-01-03-01-01-02-01.wav ... Angry
(15/2880) 03-01-03-01-02-01-01.wav ... Happy
Angry880) 03-01-03-01-02-02-01.wav ... 
(17/2880) 03-01-03-02-01-01-01.wav ... Neutral
(18/2880) 03-01-03-02-01-02-01.wav ... Happy
(19/2880) 03-01-03-02-02-01-01.wav ... Happy
(20/2880) 03-01-03-02-02-02-01.wav ... Happy
(21/2880) 03-01-04-01-01-01-01.wav ... Angry
Happy880) 03-01-04-01-01-02-01.wav ... 
(23/2880) 03-01-04-01-02-01-01.wav ... Happy
(24/2880) 03-01-04-01-

In [5]:
import json
with open(output_file, "r") as jsonfile: 
    data = json.load(jsonfile)

In [10]:
len(data)

1440